# 03- Churn Classification, Benchmarking & Explainability
### SmartTel Retention Intelligence — Module 3

**Input:** `telco_features_with_personas_latest.parquet` (produced by `02_segmentation.ipynb`)

**Scope of this notebook**:
1. Prepare the modeling feature set (and re-confirm exclusions from Modules 1–2)
2. Benchmark Logistic Regression → Random Forest → XGBoost via cross-validation
3. Evaluate on a held-out test set with ROC-AUC, PR-AUC, and calibration curves
4. Apply probability calibration to the selected model
5. Explain predictions with SHAP — globally, per persona, and per customer
6. Run a preliminary fairness check on model *output* (Module 1 checked the raw data; this checks the model's predictions)
7. Persist the calibrated model and scored dataset

> **Note:** XGBoost is used here as the boosted-tree representative to keep the benchmark focused. Swapping in CatBoost or LightGBM later would slot into the same benchmarking harness below.


## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import date
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    brier_score_loss, classification_report, confusion_matrix,
)
from xgboost import XGBClassifier
import shap

pd.set_option("display.max_columns", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

FEATURE_STORE_DIR = Path("../data/feature_store")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 2. Load Feature-Store Dataset

In [ ]:
df = pd.read_parquet(FEATURE_STORE_DIR / "telco_features_with_personas_latest.parquet")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


Shape: 7,043 rows x 27 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket,services_count,avg_monthly_revenue,contract_risk_flag,persona_id,persona_name
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-6mo,1,29.850000,1,3,New & Price-Sensitive
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48mo,3,55.573529,0,3,New & Price-Sensitive
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-6mo,3,54.075000,0,3,New & Price-Sensitive
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48mo,3,40.905556,0,3,New & Price-Sensitive
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-6mo,1,75.825000,1,3,New & Price-Sensitive
